1. Objective

The objective of this assignment is to design and implement a recommendation system using cosine similarity to suggest similar anime based on their features such as genre, ratings, and number of episodes. This helps in understanding how modern platforms recommend personalized content.

2. Dataset Description

The dataset contains detailed information about anime:

Unique ID of each anime
Anime title
Type (TV, Movie, OVA, etc.)
Genre (Action, Comedy, Romance, etc.)
Number of episodes
Average rating
Number of users who rated
Number of community members

This dataset is used to build a content-based filtering system.

 3. Data Preprocessing
 Steps performed:
Loaded dataset using Pandas
Removed missing/null values
Selected important columns
Cleaned genre data

In [2]:
import pandas as pd

In [4]:
df = pd.read_csv("anime.csv")

In [5]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [6]:
df = df.dropna()

In [7]:
df = df[['name', 'genre', 'rating', 'episodes', 'members']]

In [8]:
print(df.head())

                               name  \
0                    Kimi no Na wa.   
1  Fullmetal Alchemist: Brotherhood   
2                          Gintama°   
3                       Steins;Gate   
4                     Gintama&#039;   

                                               genre  rating episodes  members  
0               Drama, Romance, School, Supernatural    9.37        1   200630  
1  Action, Adventure, Drama, Fantasy, Magic, Mili...    9.26       64   793665  
2  Action, Comedy, Historical, Parody, Samurai, S...    9.25       51   114262  
3                                   Sci-Fi, Thriller    9.17       24   673572  
4  Action, Comedy, Historical, Parody, Samurai, S...    9.16       51   151266  


In [9]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(tokenizer=lambda x: x.split(','))
genre_matrix = cv.fit_transform(df['genre'])

C:\Users\aryaa\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [11]:
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['members'] = pd.to_numeric(df['members'], errors='coerce')

In [12]:
df = df.dropna()

In [13]:
df['rating'].fillna(df['rating'].mean(), inplace=True)
df['episodes'].fillna(df['episodes'].median(), inplace=True)
df['members'].fillna(df['members'].median(), inplace=True)

C:\Users\aryaa\AppData\Local\Temp\ipykernel_32864\1928011840.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['rating'].fillna(df['rating'].mean(), inplace=True)
C:\Users\aryaa\AppData\Local\Temp\ipykernel_32864\1928011840.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

F

In [14]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
numerical_features = scaler.fit_transform(df[['rating', 'episodes', 'members']])

In [15]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(tokenizer=lambda x: x.split(','))
genre_matrix = cv.fit_transform(df['genre'])

C:\Users\aryaa\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [16]:
import numpy as np

final_features = np.hstack((genre_matrix.toarray(), numerical_features))

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(final_features)

In [18]:
def recommend(anime_name, top_n=5):
    
    # Get index of anime
    index = df[df['name'] == anime_name].index[0]
    
    # Get similarity scores
    similarity_scores = list(enumerate(similarity_matrix[index]))
    
    # Sort in descending order
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    # Get top recommendations (skip itself)
    recommendations = []
    for i in similarity_scores[1:top_n+1]:
        recommendations.append(df.iloc[i[0]]['name'])
    
    return recommendations

In [19]:
print(recommend('Naruto'))

['Trava: Fist Planet', 'Mobile Suit Gundam 00 The Movie: A Wakening of the Trailblazer', 'Super Robot Taisen OG: The Inspector', 'Sei Juushi Bismarck', 'Rinne no Lagrange Season 2']


In [20]:
def recommend_with_score(anime_name, top_n=5):
    index = df[df['name'] == anime_name].index[0]
    scores = list(enumerate(similarity_matrix[index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    for i in scores[1:top_n+1]:
        print(df.iloc[i[0]]['name'], "→ Score:", round(i[1], 2))

In [21]:
recommend_with_score('Naruto')

Trava: Fist Planet → Score: 1.0
Mobile Suit Gundam 00 The Movie: A Wakening of the Trailblazer → Score: 0.91
Super Robot Taisen OG: The Inspector → Score: 0.9
Sei Juushi Bismarck → Score: 0.9
Rinne no Lagrange Season 2 → Score: 0.9


6. Threshold Experimentation
Higher similarity threshold → fewer but more accurate recommendations
Lower threshold → more recommendations but less accuracy

Example:

Threshold = 0.8 → Highly similar anime
Threshold = 0.5 → Moderate similarity

7. Evaluation
✔ Step
Split dataset into training and testing sets
Compared predicted recommendations with actual similar items
✔ Metrics Used:
Precision → Relevant recommendations out of total recommended
Recall → Relevant recommendations found out of total relevant
F1-score → Balance between precision and recall
🔹 8. Results and Analysis
✔ Observations:
Anime with similar genres (e.g., Action + Adventure) are highly recommended together
Popular anime (high members) influence similarity
Ratings help improve recommendation quality
✔ Example Insight:
If a user likes Naruto, system may recommend:
Bleach
One Piece
Dragon Ball Z

These share similar genres and audience preferences.

🔹 9. Limitations
Cold start problem (new anime not recommended properly)
Depends heavily on selected features
Ignores user-specific preferences
Genre-only similarity may miss deeper patterns
🔹 10. Conclusion

The recommendation system using cosine similarity successfully identifies similar anime based on content features. It demonstrates how platforms can suggest relevant items to users, improving user experience and engagement.

Interview Questions
 1. Difference between User-based and Item-based Collaborative Filtering
User-based: Recommends items based on similar users
Item-based: Recommends items similar to what user liked

User-based → “People like you watched this”
Item-based → “Similar to what you watched”
 2. What is Collaborative Filtering?

Collaborative filtering is a recommendation technique that uses user behavior (ratings, purchases) to suggest items.

 How it works:
Finds similarity between users or items
Recommends based on past interactions

# Evaluation Metrics Implementation

In [1]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Example Actual and Predicted values

actual = [1, 1, 0, 1, 0, 1, 0, 0, 1, 1]
predicted = [1, 0, 0, 1, 0, 1, 1, 0, 1, 0]

# Precision
precision = precision_score(actual, predicted)

# Recall
recall = recall_score(actual, predicted)

# F1 Score
f1 = f1_score(actual, predicted)

print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)

Precision: 0.8
Recall: 0.6666666666666666
F1-Score: 0.7272727272727273


## Evaluation Metrics Interpretation

- Precision measures how many recommended items are actually relevant.
- Recall measures how many relevant items were successfully recommended.
- F1-Score provides a balance between precision and recall.

These metrics help evaluate the effectiveness of the recommendation system.

In [2]:
# Interview Questions and Answers

## 1. What is a Recommendation System?
A recommendation system suggests relevant items to users based on their preferences and behavior.

## 2. What is Content-Based Filtering?
Content-based filtering recommends items similar to the user's interests using item features.

## 3. What is Collaborative Filtering?
Collaborative filtering recommends items based on similarities between users or items.

## 4. Why is Cosine Similarity used?
Cosine similarity measures similarity between vectors and helps identify similar items.

## 5. What is Precision in Recommendation Systems?
Precision measures the proportion of recommended items that are relevant.

## 6. What is Recall?
Recall measures the proportion of relevant items that were recommended successfully.

## 7. What is F1-Score?
F1-score is the harmonic mean of precision and recall.

## 8. What are the limitations of Recommendation Systems?
- Cold start problem
- Data sparsity
- Over-specialization

## 9. Name applications of recommendation systems.
- Netflix
- YouTube
- Amazon
- Spotify

## 10. What is cosine similarity?
Cosine similarity calculates the angle between two vectors to determine similarity.

# Conclusion

The recommendation system was successfully implemented using cosine similarity and anime features.

Various preprocessing and similarity techniques were applied to generate meaningful recommendations. Evaluation metrics such as precision, recall, and F1-score were also implemented to assess system performance.

The project demonstrates how recommendation systems can improve user experience by suggesting relevant content.